In [1]:
import pandas as pd
import math
from itertools import combinations
from mendeleev import element


def get_pauling_en(symbol):
    try:
        en = element(symbol).en_pauling
        return en
    except:
        return None


def pauling_q(el1, el2):
    chi1 = get_pauling_en(el1)
    chi2 = get_pauling_en(el2)
    if chi1 is None or chi2 is None:
        return None
    delta = abs(chi1 - chi2)
    return 1 - math.exp(-(delta ** 2) / 4)


def avg_ionicity(element_list):
    unknowns = [el for el in element_list if get_pauling_en(el) is None]
    if unknowns:
        return f"Missing: {','.join(unknowns)}"
    
    pairs = list(combinations(element_list, 2))
    values = [pauling_q(a, b) for a, b in pairs if pauling_q(a, b) is not None]
    return round(sum(values) / len(values), 4) if values else None


input_file = "Ionicity-Layered TMHs-input.xlsx"
df = pd.read_excel(input_file)


df["Average Pauling Ionicity"] = df["Elements"].apply(
    lambda x: avg_ionicity([e.strip() for e in x.split(",")])
)


output_file = "Ionicity-Layered TMHs-output.xlsx"
df.to_excel(output_file, index=False)

print(f"✅ Done! Saved to '{output_file}'")


failures = df[df["Average Pauling Ionicity"].astype(str).str.startswith("Missing")]
if not failures.empty:
    print("\n⚠️ Materials with unknown or unsupported elements:")
    print(failures[["Material", "Elements", "Average Pauling Ionicity"]])


✅ Done! Saved to 'Ionicity-Layered TMHs-output.xlsx'
